In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# SETTINGS
# =========================================================
PRED_FILE = r"C:\IDEAL_Programming\One_by_One\Home77_beh_adj\ui_pred_combined_coldstart_Edinburgh_2027-01-28_20260414_105806.csv"
REAL_FILE = r"C:\IDEAL_Programming\One_by_One\Home77_beh_adj\home77_real_profile_0128.csv"

HOME_ID = "home77"
LABEL = "home77_exact"
SEASON = "winter"
MONTH_DAY = "01-28"

MODEL_COLUMNS = {
    "rf": "rf_Wh",
    "xgb": "xgb_Wh",
    "lgbm": "lgbm_Wh",
    "ensemble": "ensemble_Wh"
}

# =========================================================
# HELPERS
# =========================================================
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    mask = np.abs(y_true) > 1e-6
    if mask.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    return mae, rmse, mape


def save_plot(hours, real, pred_dict, out_path):
    plt.figure(figsize=(10, 5))
    plt.plot(hours, real, label="Real", linewidth=2)

    for model_name, pred in pred_dict.items():
        plt.plot(hours, pred, label=model_name)

    plt.title(f"{LABEL} - cold-start ({SEASON})")
    plt.xlabel("Hour")
    plt.ylabel("Consumption (Wh)")
    plt.xticks(hours)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# =========================================================
# LOAD DATA
# =========================================================
pred_df = pd.read_csv(PRED_FILE)
real_df = pd.read_csv(REAL_FILE)

pred_df.columns = pred_df.columns.str.strip()
real_df.columns = real_df.columns.str.strip()

# =========================================================
# PREPARE PREDICTIONS
# =========================================================
if "timestamp" in pred_df.columns:
    pred_df["timestamp"] = pd.to_datetime(pred_df["timestamp"], errors="coerce", dayfirst=False)
    pred_df = pred_df.dropna(subset=["timestamp"]).copy()
    pred_df["hour"] = pred_df["timestamp"].dt.hour
    pred_df = pred_df.sort_values("hour").reset_index(drop=True)
else:
    raise ValueError("Prediction file does not contain 'timestamp' column.")

# =========================================================
# PREPARE REAL PROFILE
# =========================================================
if "hour" not in real_df.columns or "real_consumption_Wh" not in real_df.columns:
    raise ValueError("Real profile file must contain columns: 'hour' and 'real_consumption_Wh'.")

real_df = real_df.sort_values("hour").reset_index(drop=True)

# =========================================================
# ALIGN HOURS
# =========================================================
common_hours = sorted(set(pred_df["hour"]).intersection(set(real_df["hour"])))

if len(common_hours) != 24:
    print(f"Warning: common hours found = {len(common_hours)} instead of 24")

pred_df = pred_df[pred_df["hour"].isin(common_hours)].sort_values("hour").reset_index(drop=True)
real_df = real_df[real_df["hour"].isin(common_hours)].sort_values("hour").reset_index(drop=True)

real_values = real_df["real_consumption_Wh"].to_numpy(dtype=float)
hours = real_df["hour"].tolist()

if len(real_values) == 0:
    raise ValueError("No overlapping hours between prediction and real profile.")

# =========================================================
# EVALUATION
# =========================================================
results = []
pred_dict_for_plot = {}

for model_name, col in MODEL_COLUMNS.items():
    if col not in pred_df.columns:
        print(f"Skipping {model_name}: column {col} not found")
        continue

    pred_values = pred_df[col].to_numpy(dtype=float)

    if len(pred_values) != len(real_values):
        print(f"Skipping {model_name}: length mismatch")
        continue

    mae, rmse, mape = compute_metrics(real_values, pred_values)

    results.append({
        "home_id": HOME_ID,
        "label": LABEL,
        "season": SEASON,
        "month_day": MONTH_DAY,
        "model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "pred_total_kWh": pred_values.sum() / 1000.0,
        "real_total_kWh": real_values.sum() / 1000.0
    })

    pred_dict_for_plot[model_name] = pred_values

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

if results_df.empty:
    raise ValueError("No results were produced. Check columns and file paths.")

# =========================================================
# SAVE OUTPUTS
# =========================================================
out_dir = os.path.dirname(PRED_FILE)

metrics_path = os.path.join(out_dir, f"{LABEL}_coldstart_metrics.csv")
results_df.to_csv(metrics_path, index=False)

plot_path = os.path.join(out_dir, f"{LABEL}_coldstart_plot.png")
save_plot(hours, real_values, pred_dict_for_plot, plot_path)

# =========================================================
# PRINT
# =========================================================
print("=" * 80)
print("EXACT HOME COLD-START RESULTS")
print("=" * 80)
print(results_df)

print("\nSaved files:")
print(metrics_path)
print(plot_path)

EXACT HOME COLD-START RESULTS
  home_id         label  season month_day     model        MAE        RMSE  \
0  home77  home77_exact  winter     01-28        rf  62.655133   79.555761   
1  home77  home77_exact  winter     01-28  ensemble  70.370732   89.757278   
2  home77  home77_exact  winter     01-28       xgb  74.720991   98.705651   
3  home77  home77_exact  winter     01-28      lgbm  77.732239  101.458084   

        MAPE  pred_total_kWh  real_total_kWh  
0  34.867499        5.002331           5.584  
1  35.638823        4.834383           5.584  
2  35.359519        4.692202           5.584  
3  38.096324        4.752633           5.584  

Saved files:
C:\IDEAL_Programming\One_by_One\Home77_beh_adj\home77_exact_coldstart_metrics.csv
C:\IDEAL_Programming\One_by_One\Home77_beh_adj\home77_exact_coldstart_plot.png
